# Prompt Engineering - Start Here

---

Prompt engineering is the fastest way to improve LLM output quality, but it is not magic and it is not a substitute for system design. This phase teaches how to give models clearer instructions, better context, and more reliable task framing.

---

## Why This Phase Matters

Good prompting improves quality, consistency, and usefulness immediately. It also teaches you how models respond to constraints, examples, roles, structure, and reasoning hints. That judgment matters even if you later move into RAG, fine-tuning, or agent workflows.

## How To Use This Phase Well

- Treat prompting as experimentation, not as one-shot writing.
- Compare vague prompts against constrained prompts and observe the difference.
- Notice where prompt improvements stop helping and you need retrieval, tooling, or evaluation instead.
- Keep the examples practical: extraction, summarization, classification, reasoning, and workflow design.

## Prerequisites

- Python basics (Phase 01)
- Some exposure to LLM APIs or notebook-based experimentation
- Earlier tokenization and embeddings phases help, but they are not required to begin this notebook

## Recommended Goal

By the end of this phase, you should know how to write clearer prompts, use examples effectively, structure outputs, and recognize when prompt engineering is enough versus when you need a broader system solution.

### Setup

Before working with LLMs programmatically, you need an API client and a reusable helper function. The `OpenAI` client handles authentication and HTTP communication with the API, while the `ask_llm()` wrapper keeps each example focused on prompting rather than request plumbing. In May 2026, the clean default for new OpenAI examples is the **Responses API** with a fast general-purpose model like `gpt-4o-mini`. The `temperature` parameter controls randomness: 0.0 is useful for extraction and classification, while higher values increase variation for brainstorming and creative tasks. Loading the API key from a `.env` file via `python-dotenv` keeps credentials out of your code.

In [ ]:
# Install required packages
# !pip install openai anthropic langchain python-dotenv

In [ ]:
import os
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv()
client = OpenAI(api_key=os.getenv('OPENAI_API_KEY'))

def ask_llm(prompt, model="gpt-4o-mini", temperature=0.7):
    """Helper function to query an LLM with the Responses API."""
    response = client.responses.create(
        model=model,
        input=prompt,
        temperature=temperature
    )
    return response.output_text

### Example 1: Basic vs. Improved Prompt

The single most impactful technique in prompt engineering is **specificity**. A vague prompt like "Tell me about Python" gives the model too many degrees of freedom -- it might discuss the language's history, syntax, ecosystem, or philosophy. A well-structured prompt constrains the scope (list comprehensions), specifies the audience (beginner), defines the format (numbered list), and sets a length limit (150 words). The difference in output quality is dramatic, and this applies to every LLM task from summarization to code generation.

In [ ]:
# ❌ Vague prompt
bad_prompt = "Tell me about Python"

# ✅ Specific prompt
good_prompt = """
Explain Python's list comprehensions to a beginner programmer.
Include:
1. What they are (1-2 sentences)
2. Basic syntax
3. One simple example
4. One common use case

Keep the explanation under 150 words.
"""

print("=== Vague Prompt ===")
print(ask_llm(bad_prompt)[:200], "...\n")

print("=== Specific Prompt ===")
print(ask_llm(good_prompt))

### Example 2: Few-Shot Learning

**Few-shot prompting** provides input-output examples directly in the prompt, teaching the model the desired behavior through demonstration rather than description. By showing two examples of sentence-to-JSON conversion, the model learns the exact output schema, key names, and extraction logic without any fine-tuning. This technique leverages the model's in-context learning ability -- a property where transformers can infer patterns from examples provided at inference time. Few-shot prompting is especially effective for structured extraction, classification, and format conversion tasks.

In [ ]:
few_shot_prompt = """
Convert the following sentences to JSON format.

Example 1:
Input: "John lives in New York and works as a doctor."
Output: {"name": "John", "city": "New York", "occupation": "doctor"}

Example 2:
Input: "Sarah is from London and she is a teacher."
Output: {"name": "Sarah", "city": "London", "occupation": "teacher"}

Now convert this:
Input: "Mike lives in Tokyo and works as an engineer."
Output:
"""

print(ask_llm(few_shot_prompt, temperature=0))

### Example 3: Chain-of-Thought Reasoning

**Chain-of-thought (CoT)** prompting improves accuracy on multi-step reasoning tasks by asking the model to show its work. Adding "Let's solve this step by step" to a math problem causes the model to decompose the problem into intermediate steps, dramatically reducing errors. The intuition is that LLMs are autoregressive -- each token is conditioned on all previous tokens -- so generating intermediate reasoning steps gives the model a "scratch pad" that guides subsequent tokens toward the correct answer. CoT is most beneficial for arithmetic, logic, commonsense reasoning, and any task requiring more than one inference step.

In [ ]:
# Without CoT
simple_prompt = """
A store had 20 apples. They sold 12 apples in the morning and 5 in the afternoon.
Then they received a delivery of 15 apples. How many apples do they have now?
"""

# With CoT
cot_prompt = simple_prompt + "\nLet's solve this step by step:"

print("=== Without Chain-of-Thought ===")
print(ask_llm(simple_prompt, temperature=0), "\n")

print("=== With Chain-of-Thought ===")
print(ask_llm(cot_prompt, temperature=0))

### Example 4: System Prompts

The **system prompt** is a special message that sets the model's persona, behavior constraints, and response style for the entire conversation. Unlike user messages, the system prompt persists across turns and acts as persistent instructions. By changing the system prompt from "computer science professor" to "patient teacher for beginners," you get fundamentally different explanations of the same concept -- technical depth versus accessible analogies. In production applications, system prompts define the core personality and guardrails of your AI feature, making them one of the most important design decisions.

In [ ]:
def ask_with_system(system_prompt, user_prompt, model="gpt-4o-mini", temperature=0.7):
    """Use instructions plus user input to shape the response."""
    response = client.responses.create(
        model=model,
        instructions=system_prompt,
        input=user_prompt,
        temperature=temperature
    )
    return response.output_text

# Example: Expert vs. Beginner explanations
question = "What is recursion in programming?"

expert_system = "You are a computer science professor. Explain concepts with technical precision and examples."
beginner_system = "You are a patient teacher for absolute beginners. Use simple language and everyday analogies."

print("=== Expert Mode ===")
print(ask_with_system(expert_system, question), "\n")

print("=== Beginner Mode ===")
print(ask_with_system(beginner_system, question))

### Example 5: Structured Output

Requesting **structured output** (JSON, XML, CSV) transforms an LLM from a free-text generator into a reliable data extraction tool. The prompt below specifies an exact JSON schema with field names, value constraints, and data types, then instructs the model to return only JSON with no surrounding text. The result can be parsed with `json.loads()` and consumed by downstream code. This technique is the foundation of LLM-powered data pipelines, where unstructured text (reviews, emails, documents) is systematically converted into structured records for databases and analytics.

In [ ]:
structured_prompt = """
Analyze the following product review and extract information in this exact JSON format:

{
  "sentiment": "positive/negative/neutral",
  "rating_guess": "1-5",
  "key_points": ["point1", "point2"],
  "mentioned_features": ["feature1", "feature2"]
}

Review: "This laptop is amazing! Super fast processor and the battery lasts all day. 
Only complaint is it's a bit heavy to carry around."

Return ONLY the JSON, no other text:
"""

result = ask_llm(structured_prompt, temperature=0)
print(result)

# Parse as JSON
import json
data = json.loads(result)
print("\nParsed data:", data)

## Key Takeaways

### 1. Be Specific

Clear instructions, explicit output formats, and well-scoped constraints usually outperform vague requests.

### 2. Use Examples When Format Matters

Few-shot prompting is especially useful for classification, extraction, and structured generation tasks.

### 3. Add Reasoning Carefully

Step-by-step prompting can improve complex tasks, but it is not necessary for everything and should be evaluated, not assumed.

### 4. Set Context Intentionally

System prompts, role framing, and task instructions shape the model's behavior across the whole interaction.

### 5. Know the Limits of Prompting

Prompt engineering helps a lot, but it does not replace retrieval, tools, fine-tuning, or evaluation when the problem requires them.

## What Comes Next

1. Complete the notebook sequence from basic prompting to more advanced reasoning patterns.
2. Use what you learn here in RAG, agents, and evaluation phases rather than treating prompting as a standalone skill.
3. When prompts stop being enough, move to retrieval, tooling, or fine-tuning instead of endlessly tweaking wording.

Prompt engineering is best understood as a practical control layer for LLM systems, not as the only optimization lever.